# W1 Day3 (04/09 周三): PyTorch Tensor + Autograd

## 学习目标
- **理论 (1-1.5h)**: Storage / Stride / View / contiguous; 计算图 DAG; 反向传播链式法则
- **实践 (2h)**: Tensor 内存实验 + 手写 3 层网络反向传播 + PyTorch 验证
- **产出物**: 反向传播实验 notebook

---

## 目录
1. [Tensor 基础: 创建与数据类型](#1)
2. [Storage / Stride / View 内存模型](#2)
3. [contiguous 与内存布局](#3)
4. [Autograd: 计算图 DAG](#4)
5. [反向传播: 链式法则推导](#5)
6. [手写 3 层网络反向传播 (纯 NumPy)](#6)
7. [PyTorch Autograd 验证](#7)
8. [Autograd 进阶: hook / detach / no_grad / grad 累积](#8)
9. [常见陷阱与调试技巧](#9)
10. [思考题 & 总结](#10)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 1. Tensor 基础: 创建与数据类型 <a id='1'></a>

Tensor 是 PyTorch 的核心数据结构，类似于 NumPy ndarray，但支持:
- GPU 加速
- 自动微分 (Autograd)

In [ ]:
print("=" * 60)
print("Tensor 创建方式")
print("=" * 60)

# 从 Python 数据创建
t1 = torch.tensor([1, 2, 3])               # 从列表
t2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]]) # 2D
t3 = torch.from_numpy(np.array([5, 6, 7]))  # 从 NumPy (共享内存!)

# 工厂函数
t4 = torch.zeros(3, 4)           # 全零
t5 = torch.ones(2, 3)            # 全一
t6 = torch.randn(3, 3)           # 标准正态
t7 = torch.arange(0, 10, 2)      # 等差序列
t8 = torch.eye(3)                # 单位矩阵
t9 = torch.empty(2, 3)           # 未初始化 (速度快)
t10 = torch.full((2, 3), 3.14)   # 填充常数

# like 系列: 继承 shape 和 dtype
t11 = torch.zeros_like(t6)
t12 = torch.randn_like(t6)

print(f"tensor:     {t1} (dtype={t1.dtype})")
print(f"2D tensor:  shape={t2.shape}, dtype={t2.dtype}")
print(f"from numpy: {t3} (dtype={t3.dtype})")
print(f"randn:      shape={t6.shape}")
print(f"arange:     {t7}")

In [ ]:
# 数据类型
print("\n" + "=" * 60)
print("数据类型 (dtype)")
print("=" * 60)

dtypes = [
    ("torch.float32 (默认)", torch.float32, 4),
    ("torch.float16 (半精度)", torch.float16, 2),
    ("torch.bfloat16 (脑浮点)", torch.bfloat16, 2),
    ("torch.float64 (双精度)", torch.float64, 8),
    ("torch.int32", torch.int32, 4),
    ("torch.int64 (默认整型)", torch.int64, 8),
    ("torch.bool", torch.bool, 1),
]

for name, dtype, nbytes in dtypes:
    t = torch.ones(1000, dtype=dtype)
    print(f"  {name:30s}: {nbytes} bytes/element, 1000个={t.nbytes:>5d} bytes")

# 类型转换
x = torch.randn(3)
print(f"\n类型转换:")
print(f"  float32: {x}")
print(f"  float16: {x.half()}")
print(f"  int:     {x.int()}")
print(f"  bool:    {(x > 0)}")

# NumPy 互转 (共享内存!)
print("\n--- NumPy 互转 (共享内存) ---")
np_arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(np_arr)
np_arr[0] = 99.0
print(f"  修改 numpy 后 tensor 也变: {t}  ← 共享内存!")

---
## 2. Storage / Stride / View 内存模型 <a id='2'></a>

### 2.1 核心概念

PyTorch Tensor 由两部分组成:

1. **Storage**: 一维连续内存块，存放实际数据
2. **Metadata**: shape, stride, offset → 决定如何解释 Storage

```
Storage (一维): [1, 2, 3, 4, 5, 6]

shape=(2,3), stride=(3,1):     shape=(3,2), stride=(2,1):
  [[1, 2, 3],                    [[1, 2],
   [4, 5, 6]]                     [3, 4],
                                  [5, 6]]
```

**Stride**: 每个维度移动一步需要跳过多少个元素。

In [ ]:
print("=" * 60)
print("Storage / Stride / View")
print("=" * 60)

# 创建一个 2D tensor
x = torch.arange(12).reshape(3, 4)
print(f"x =\n{x}")
print(f"  shape:   {x.shape}")
print(f"  stride:  {x.stride()}  ← (行跳4个元素, 列跳1个元素)")
print(f"  storage: {x.storage().tolist()}")
print(f"  is_contiguous: {x.is_contiguous()}")
print(f"  data_ptr: {x.data_ptr()}")

In [ ]:
# 2.2 View: 共享 Storage，改变 shape
print("\n--- View: 共享 Storage ---")

x = torch.arange(12).reshape(3, 4)
y = x.view(4, 3)   # view: 改变形状，共享内存
z = x.view(2, 6)   # 另一种 view
w = x.view(-1)     # 展平

print(f"x.shape={x.shape}, stride={x.stride()}")
print(f"y.shape={y.shape}, stride={y.stride()}")
print(f"z.shape={z.shape}, stride={z.stride()}")
print(f"w.shape={w.shape}, stride={w.stride()}")

# 验证共享内存
print(f"\n共享内存: x.data_ptr() == y.data_ptr(): {x.data_ptr() == y.data_ptr()}")
print(f"共享 storage: x.storage().data_ptr() == y.storage().data_ptr(): "
      f"{x.storage().data_ptr() == y.storage().data_ptr()}")

# 修改 y 会影响 x
y[0, 0] = 999
print(f"\ny[0,0]=999 后 x[0,0]={x[0,0]}  ← 同一块内存!")
x[0, 0] = 0  # 恢复

In [ ]:
# 2.3 Stride 详解: 如何定位元素
print("\n" + "=" * 60)
print("Stride 详解: element[i,j] = storage[offset + i*stride[0] + j*stride[1]]")
print("=" * 60)

x = torch.arange(12).reshape(3, 4)
print(f"x = {x.tolist()}")
print(f"storage = {x.storage().tolist()}")
print(f"stride = {x.stride()}, offset = {x.storage_offset()}")
print()

# 手动计算几个元素的位置
for i in range(3):
    for j in range(4):
        idx = x.storage_offset() + i * x.stride()[0] + j * x.stride()[1]
        val = x.storage()[idx]
        print(f"  x[{i},{j}] = storage[{idx}] = {val}  (验证: {x[i,j].item()})")

In [ ]:
# 2.4 转置: 只改变 stride，不移动数据
print("\n" + "=" * 60)
print("转置: 交换 stride，零拷贝")
print("=" * 60)

x = torch.arange(6).reshape(2, 3)
xt = x.T  # 转置

print(f"x:  shape={x.shape},  stride={x.stride()},  contiguous={x.is_contiguous()}")
print(f"xT: shape={xt.shape}, stride={xt.stride()}, contiguous={xt.is_contiguous()}")
print(f"共享内存: {x.data_ptr() == xt.data_ptr()}")
print(f"\nx  = {x.tolist()}")
print(f"xT = {xt.tolist()}")
print(f"storage (相同): {x.storage().tolist()}")

print("\n💡 转置只交换了 stride (3,1) → (1,3)，数据完全没动!")

In [ ]:
# 2.5 切片: offset + stride
print("\n" + "=" * 60)
print("切片: 改变 offset + shape + stride")
print("=" * 60)

x = torch.arange(20).reshape(4, 5)
print(f"x = \n{x}")

s = x[1:3, 2:5]  # 切片
print(f"\nx[1:3, 2:5] = \n{s}")
print(f"  shape: {s.shape}")
print(f"  stride: {s.stride()} (与 x 相同!)")
print(f"  storage_offset: {s.storage_offset()} (跳过了前面的元素)")
print(f"  contiguous: {s.is_contiguous()}")
print(f"  共享内存: {s.data_ptr() != x.data_ptr()} (但指向同一 storage 的不同位置)")
print(f"  同一 storage: {x.storage().data_ptr() == s.storage().data_ptr()}")

---
## 3. contiguous 与内存布局 <a id='3'></a>

### 什么时候需要 contiguous？

- `view()` 要求 tensor 是 contiguous 的
- 某些底层操作 (如传给 C 库) 需要连续内存
- 非 contiguous → `contiguous()` 会**复制数据**到新的连续内存

In [ ]:
print("=" * 60)
print("contiguous 实验")
print("=" * 60)

x = torch.arange(12).reshape(3, 4)
xt = x.T  # 转置后不再 contiguous

print(f"x  contiguous: {x.is_contiguous()}, stride: {x.stride()}")
print(f"xT contiguous: {xt.is_contiguous()}, stride: {xt.stride()}")

# view 在非 contiguous 上会失败
try:
    xt.view(-1)  # 报错!
except RuntimeError as e:
    print(f"\n✗ xt.view(-1) 失败: {e}")

# 解决方案 1: contiguous() + view()
xt_c = xt.contiguous()
print(f"\nxt.contiguous():")
print(f"  contiguous: {xt_c.is_contiguous()}, stride: {xt_c.stride()}")
print(f"  新内存: {xt_c.data_ptr() != xt.data_ptr()}  ← 发生了数据复制!")
flat = xt_c.view(-1)
print(f"  view(-1): {flat}")

# 解决方案 2: reshape() (自动处理)
flat2 = xt.reshape(-1)  # 自动判断是否需要 contiguous
print(f"\nxt.reshape(-1): {flat2}")
print(f"  与 contiguous().view() 相同: {torch.equal(flat, flat2)}")

print("\n💡 view() 不复制，但要求 contiguous")
print("💡 reshape() 自动处理: contiguous 时等于 view，否则等于 contiguous().view()")
print("💡 性能敏感时用 view() 避免意外复制")

In [ ]:
# 内存布局: Row-major (C) vs Column-major (Fortran)
print("\n" + "=" * 60)
print("内存布局可视化")
print("=" * 60)

x = torch.arange(12).reshape(3, 4)
xt = x.T

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 原始 tensor
ax = axes[0]
storage = x.storage().tolist()
ax.bar(range(len(storage)), storage, color='steelblue')
ax.set_title(f'x (3×4)\nstride={x.stride()}, contiguous={x.is_contiguous()}')
ax.set_xlabel('Storage index')
ax.set_ylabel('Value')

# 转置 (共享 storage)
ax = axes[1]
# 按转置后的逻辑顺序标注
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12'] * 3
ax.bar(range(len(storage)), storage, color=colors[:len(storage)])
ax.set_title(f'x.T (4×3)\nstride={xt.stride()}, contiguous={xt.is_contiguous()}')
ax.set_xlabel('Storage index (不连续读取!)')

# contiguous 后的新 storage
ax = axes[2]
xt_c = xt.contiguous()
storage_new = xt_c.storage().tolist()
ax.bar(range(len(storage_new)), storage_new, color='seagreen')
ax.set_title(f'x.T.contiguous() (4×3)\nstride={xt_c.stride()}, 新内存')
ax.set_xlabel('Storage index (连续读取)')

plt.suptitle('Storage 布局对比', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Autograd: 计算图 DAG <a id='4'></a>

### 4.1 计算图的概念

PyTorch 使用**动态计算图 (Define-by-Run)**:
- 每次前向传播时构建计算图
- 反向传播后图被释放
- 图是 DAG (有向无环图): 节点=运算, 边=Tensor

### 4.2 requires_grad

- `requires_grad=True`: 告诉 PyTorch 追踪该 tensor 的所有运算
- 图中的每个中间 tensor 都记录了创建它的 `grad_fn`

In [ ]:
print("=" * 60)
print("Autograd: 计算图构建")
print("=" * 60)

# 创建需要梯度的 tensor
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

print(f"x = {x}, requires_grad = {x.requires_grad}")
print(f"y = {y}, requires_grad = {y.requires_grad}")

# 前向计算: z = x² + 2xy + y²  = (x+y)²
a = x ** 2       # a = x²
b = 2 * x * y    # b = 2xy
c = y ** 2       # c = y²
z = a + b + c    # z = x² + 2xy + y²

print(f"\n前向计算:")
print(f"  a = x² = {a.item():.1f}, grad_fn = {a.grad_fn}")
print(f"  b = 2xy = {b.item():.1f}, grad_fn = {b.grad_fn}")
print(f"  c = y² = {c.item():.1f}, grad_fn = {c.grad_fn}")
print(f"  z = a+b+c = {z.item():.1f}, grad_fn = {z.grad_fn}")

# 反向传播
z.backward()

# dz/dx = 2x + 2y = 2(2) + 2(3) = 10
# dz/dy = 2x + 2y = 2(2) + 2(3) = 10
print(f"\n反向传播:")
print(f"  dz/dx = {x.grad.item():.1f}  (理论: 2x+2y = {2*2+2*3})")
print(f"  dz/dy = {y.grad.item():.1f}  (理论: 2x+2y = {2*2+2*3})")

In [ ]:
# 4.2 计算图可视化
print("\n" + "=" * 60)
print("计算图结构追踪")
print("=" * 60)

def trace_graph(tensor, indent=0):
    """递归打印计算图结构"""
    prefix = "  " * indent
    if tensor.grad_fn is not None:
        print(f"{prefix}├─ {tensor.grad_fn.__class__.__name__}")
        for child, _ in tensor.grad_fn.next_functions:
            if child is not None:
                # AccumulateGrad 是叶子节点
                if child.__class__.__name__ == 'AccumulateGrad':
                    print(f"{prefix}│  └─ Leaf: {child.variable.data.item():.1f} "
                          f"(requires_grad={child.variable.requires_grad})")
                else:
                    # 创建一个临时 tensor 来继续追踪
                    print(f"{prefix}│  └─ {child.__class__.__name__}")
    elif tensor.requires_grad:
        print(f"{prefix}└─ Leaf tensor: {tensor.data.item():.1f}")
    else:
        print(f"{prefix}└─ Constant: {tensor.data.item():.1f}")

# 构建一个稍复杂的图
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = torch.sin(x) * torch.exp(y) + x ** 2

print(f"z = sin(x) * exp(y) + x²")
print(f"z = sin(2) * exp(3) + 4 = {z.item():.4f}")
print(f"\n计算图:")
trace_graph(z)

z.backward()
# dz/dx = cos(x)*exp(y) + 2x = cos(2)*exp(3) + 4
expected_dx = math.cos(2) * math.exp(3) + 2 * 2
# dz/dy = sin(x)*exp(y)
expected_dy = math.sin(2) * math.exp(3)

print(f"\n梯度:")
print(f"  dz/dx = {x.grad.item():.4f} (理论: {expected_dx:.4f})")
print(f"  dz/dy = {y.grad.item():.4f} (理论: {expected_dy:.4f})")

---
## 5. 反向传播: 链式法则推导 <a id='5'></a>

### 5.1 链式法则 (Chain Rule)

如果 $z = f(g(x))$，则:
$$\frac{dz}{dx} = \frac{dz}{dg} \cdot \frac{dg}{dx}$$

反向传播就是**从输出到输入，逐层应用链式法则**。

### 5.2 一个 2 层网络的完整推导

```
前向:  x → [W1, b1] → h = ReLU(W1·x + b1) → [W2, b2] → y = W2·h + b2 → L = MSE(y, target)

反向:
  ∂L/∂y     = 2(y - target) / N
  ∂L/∂W2    = ∂L/∂y · h^T
  ∂L/∂b2    = ∂L/∂y
  ∂L/∂h     = W2^T · ∂L/∂y
  ∂L/∂z1    = ∂L/∂h ⊙ ReLU'(z1)     (z1 = W1·x + b1)
  ∂L/∂W1    = ∂L/∂z1 · x^T
  ∂L/∂b1    = ∂L/∂z1
```

In [ ]:
# 5.1 单个运算的梯度验证
print("=" * 60)
print("常见运算的梯度")
print("=" * 60)

def check_grad(expr_str, x_val, analytical_grad):
    """验证自动微分与解析梯度一致"""
    x = torch.tensor(x_val, requires_grad=True, dtype=torch.float64)
    y = eval(expr_str, {'x': x, 'torch': torch})
    y.backward()
    
    auto_grad = x.grad.item()
    match = abs(auto_grad - analytical_grad) < 1e-6
    symbol = "✅" if match else "❌"
    print(f"  {symbol} y = {expr_str:25s} | dy/dx at x={x_val}: "
          f"auto={auto_grad:.6f}, 解析={analytical_grad:.6f}")

x_val = 2.0

check_grad("x ** 2",           x_val, 2 * x_val)                    # 2x
check_grad("x ** 3",           x_val, 3 * x_val**2)                 # 3x²
check_grad("torch.sin(x)",     x_val, math.cos(x_val))              # cos(x)
check_grad("torch.exp(x)",     x_val, math.exp(x_val))              # exp(x)
check_grad("torch.log(x)",     x_val, 1.0 / x_val)                  # 1/x
check_grad("torch.sigmoid(x)", x_val,                                # σ(x)(1-σ(x))
           torch.sigmoid(torch.tensor(x_val)).item() * 
           (1 - torch.sigmoid(torch.tensor(x_val)).item()))
check_grad("torch.tanh(x)",    x_val,                                # 1-tanh²(x)
           1 - math.tanh(x_val)**2)
check_grad("torch.relu(x)",    x_val, 1.0 if x_val > 0 else 0.0)   # 0 or 1
check_grad("1.0 / x",          x_val, -1.0 / x_val**2)              # -1/x²

---
## 6. 手写 3 层网络反向传播 (纯 NumPy) <a id='6'></a>

完全不使用 PyTorch 的 autograd，手动推导并实现每一层的梯度。

网络结构: `Input(D) → Linear(H1) → ReLU → Linear(H2) → ReLU → Linear(C) → Softmax → CE Loss`

In [ ]:
class ManualNeuralNetwork:
    """
    手写 3 层全连接网络 (纯 NumPy)
    包含完整的前向传播和反向传播
    """
    def __init__(self, input_dim, hidden1, hidden2, output_dim):
        # Xavier 初始化
        self.W1 = np.random.randn(input_dim, hidden1) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden1)
        self.W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2.0 / hidden1)
        self.b2 = np.zeros(hidden2)
        self.W3 = np.random.randn(hidden2, output_dim) * np.sqrt(2.0 / hidden2)
        self.b3 = np.zeros(output_dim)
        
        # 缓存前向传播的中间值 (反向传播需要)
        self.cache = {}
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_backward(self, x):
        """ReLU 导数: x>0 时为1，否则为0"""
        return (x > 0).astype(float)
    
    def softmax(self, x):
        """数值稳定的 softmax"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def cross_entropy_loss(self, probs, targets):
        """交叉熵损失"""
        N = len(targets)
        log_probs = -np.log(probs[np.arange(N), targets] + 1e-8)
        return np.mean(log_probs)
    
    def forward(self, X):
        """
        前向传播
        X: (N, D) 输入
        返回: (N, C) 概率分布
        """
        # Layer 1: Linear + ReLU
        z1 = X @ self.W1 + self.b1           # (N, H1)
        a1 = self.relu(z1)                    # (N, H1)
        
        # Layer 2: Linear + ReLU
        z2 = a1 @ self.W2 + self.b2           # (N, H2)
        a2 = self.relu(z2)                    # (N, H2)
        
        # Layer 3: Linear + Softmax
        z3 = a2 @ self.W3 + self.b3           # (N, C)
        probs = self.softmax(z3)              # (N, C)
        
        # 缓存
        self.cache = {
            'X': X, 'z1': z1, 'a1': a1,
            'z2': z2, 'a2': a2, 'z3': z3, 'probs': probs
        }
        
        return probs
    
    def backward(self, targets):
        """
        反向传播: 手动计算每层梯度
        targets: (N,) 类别索引
        返回: 所有参数的梯度字典
        """
        N = len(targets)
        X = self.cache['X']
        z1, a1 = self.cache['z1'], self.cache['a1']
        z2, a2 = self.cache['z2'], self.cache['a2']
        z3, probs = self.cache['z3'], self.cache['probs']
        
        # ========== Layer 3 (输出层) ==========
        # ∂L/∂z3 = probs - one_hot(targets)  (softmax + CE 的梯度简化!)
        dz3 = probs.copy()  # (N, C)
        dz3[np.arange(N), targets] -= 1
        dz3 /= N
        
        # ∂L/∂W3 = a2^T · ∂L/∂z3
        dW3 = a2.T @ dz3   # (H2, C)
        db3 = np.sum(dz3, axis=0)  # (C,)
        
        # ========== Layer 2 ==========
        # ∂L/∂a2 = ∂L/∂z3 · W3^T
        da2 = dz3 @ self.W3.T  # (N, H2)
        
        # ∂L/∂z2 = ∂L/∂a2 ⊙ ReLU'(z2)
        dz2 = da2 * self.relu_backward(z2)  # (N, H2)
        
        # ∂L/∂W2 = a1^T · ∂L/∂z2
        dW2 = a1.T @ dz2   # (H1, H2)
        db2 = np.sum(dz2, axis=0)  # (H2,)
        
        # ========== Layer 1 ==========
        # ∂L/∂a1 = ∂L/∂z2 · W2^T
        da1 = dz2 @ self.W2.T  # (N, H1)
        
        # ∂L/∂z1 = ∂L/∂a1 ⊙ ReLU'(z1)
        dz1 = da1 * self.relu_backward(z1)  # (N, H1)
        
        # ∂L/∂W1 = X^T · ∂L/∂z1
        dW1 = X.T @ dz1    # (D, H1)
        db1 = np.sum(dz1, axis=0)  # (H1,)
        
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2, 'dW3': dW3, 'db3': db3}
    
    def update(self, grads, lr=0.01):
        """SGD 更新"""
        self.W1 -= lr * grads['dW1']
        self.b1 -= lr * grads['db1']
        self.W2 -= lr * grads['dW2']
        self.b2 -= lr * grads['db2']
        self.W3 -= lr * grads['dW3']
        self.b3 -= lr * grads['db3']


print("=" * 60)
print("手写 3 层网络 (纯 NumPy)")
print("=" * 60)

# 生成数据
np.random.seed(42)
N, D, C = 200, 10, 4
X_train = np.random.randn(N, D)
y_train = np.random.randint(0, C, N)

# 创建网络
net = ManualNeuralNetwork(input_dim=D, hidden1=32, hidden2=16, output_dim=C)

# 训练
losses = []
for epoch in range(200):
    probs = net.forward(X_train)
    loss = net.cross_entropy_loss(probs, y_train)
    grads = net.backward(y_train)
    net.update(grads, lr=0.1)
    losses.append(loss)
    
    if (epoch + 1) % 50 == 0:
        preds = np.argmax(probs, axis=1)
        acc = np.mean(preds == y_train) * 100
        print(f"  Epoch {epoch+1:3d} | Loss: {loss:.4f} | Acc: {acc:.1f}%")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('手写 3 层网络训练 Loss (纯 NumPy)')
plt.tight_layout()
plt.show()

---
## 7. PyTorch Autograd 验证 <a id='7'></a>

用 PyTorch 的 autograd 验证我们手写的梯度是否正确。

In [ ]:
print("=" * 60)
print("PyTorch Autograd 验证手写梯度")
print("=" * 60)

# 用相同的数据和权重
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.long)

# 重新初始化一个相同结构的手写网络
np.random.seed(123)
manual_net = ManualNeuralNetwork(input_dim=D, hidden1=32, hidden2=16, output_dim=C)

# PyTorch 网络使用相同权重
W1_t = torch.tensor(manual_net.W1, dtype=torch.float32, requires_grad=True)
b1_t = torch.tensor(manual_net.b1, dtype=torch.float32, requires_grad=True)
W2_t = torch.tensor(manual_net.W2, dtype=torch.float32, requires_grad=True)
b2_t = torch.tensor(manual_net.b2, dtype=torch.float32, requires_grad=True)
W3_t = torch.tensor(manual_net.W3, dtype=torch.float32, requires_grad=True)
b3_t = torch.tensor(manual_net.b3, dtype=torch.float32, requires_grad=True)

# PyTorch 前向 (与手写完全相同的计算)
z1 = X_t @ W1_t + b1_t
a1 = torch.relu(z1)
z2 = a1 @ W2_t + b2_t
a2 = torch.relu(z2)
z3 = a2 @ W3_t + b3_t
loss_pt = F.cross_entropy(z3, y_t)

# 手写前向
probs = manual_net.forward(X_train)
loss_manual = manual_net.cross_entropy_loss(probs, y_train)

print(f"Loss 对比: 手写={loss_manual:.6f}, PyTorch={loss_pt.item():.6f}")

# PyTorch 反向
loss_pt.backward()

# 手写反向
grads_manual = manual_net.backward(y_train)

# 对比梯度
print(f"\n梯度对比 (最大绝对误差):")
grad_pairs = [
    ('dW1', W1_t.grad, grads_manual['dW1']),
    ('db1', b1_t.grad, grads_manual['db1']),
    ('dW2', W2_t.grad, grads_manual['dW2']),
    ('db2', b2_t.grad, grads_manual['db2']),
    ('dW3', W3_t.grad, grads_manual['dW3']),
    ('db3', b3_t.grad, grads_manual['db3']),
]

all_match = True
for name, pt_grad, manual_grad in grad_pairs:
    max_diff = (pt_grad.numpy() - manual_grad).max()
    match = abs(max_diff) < 1e-5
    all_match = all_match and match
    symbol = "✅" if match else "❌"
    print(f"  {symbol} {name}: max_diff = {max_diff:.2e}")

if all_match:
    print("\n🎉 所有手写梯度与 PyTorch autograd 一致!")

In [ ]:
# 数值梯度检查 (有限差分法)
print("\n" + "=" * 60)
print("数值梯度检查 (有限差分法)")
print("=" * 60)

def numerical_gradient(f, x, eps=1e-5):
    """用中心差分法计算数值梯度"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = x[idx]
        
        x[idx] = old_val + eps
        f_plus = f()
        
        x[idx] = old_val - eps
        f_minus = f()
        
        grad[idx] = (f_plus - f_minus) / (2 * eps)
        x[idx] = old_val
        it.iternext()
    return grad

# 只检查 W3 (全量检查太慢)
np.random.seed(123)
small_net = ManualNeuralNetwork(input_dim=5, hidden1=8, hidden2=4, output_dim=3)
X_small = np.random.randn(10, 5)
y_small = np.random.randint(0, 3, 10)

# 解析梯度
small_net.forward(X_small)
analytical_grads = small_net.backward(y_small)

# 数值梯度
def loss_fn():
    probs = small_net.forward(X_small)
    return small_net.cross_entropy_loss(probs, y_small)

for param_name, param in [('W3', small_net.W3), ('W2', small_net.W2), ('W1', small_net.W1)]:
    num_grad = numerical_gradient(loss_fn, param, eps=1e-5)
    ana_grad = analytical_grads[f'd{param_name}']
    
    # 相对误差
    rel_error = np.abs(num_grad - ana_grad) / (np.abs(num_grad) + np.abs(ana_grad) + 1e-8)
    max_rel_err = rel_error.max()
    symbol = "✅" if max_rel_err < 1e-5 else "❌"
    print(f"  {symbol} {param_name}: max relative error = {max_rel_err:.2e}")

print("\n💡 数值梯度检查是调试自定义反向传播的标准方法")

---
## 8. Autograd 进阶 <a id='8'></a>

掌握 autograd 的关键机制和常见用法。

In [ ]:
# 8.1 梯度累积 (重要陷阱!)
print("=" * 60)
print("梯度累积 (必须手动清零!)")
print("=" * 60)

x = torch.tensor(3.0, requires_grad=True)
print(f"初始 grad: {x.grad}")

# 第一次反向
y = x ** 2  # dy/dx = 2x = 6
y.backward()
print(f"第1次 backward: grad = {x.grad.item():.1f}  (正确: 6)")

# 第二次反向 (不清零)
y = x ** 2
y.backward()
print(f"第2次 backward (不清零): grad = {x.grad.item():.1f}  (错误! 6+6=12)")

# 正确做法: 清零
x.grad.zero_()
y = x ** 2
y.backward()
print(f"清零后: grad = {x.grad.item():.1f}  (正确: 6)")

print("\n💡 optimizer.zero_grad() 就是在做这件事!")
print("💡 梯度累积可以用来模拟大 batch: 多次 backward 后再 step")

In [ ]:
# 8.2 detach 和 no_grad
print("\n" + "=" * 60)
print("detach() 和 torch.no_grad()")
print("=" * 60)

# detach: 从计算图中分离
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y.detach()  # z 不再有 grad_fn
print(f"y.requires_grad: {y.requires_grad}, grad_fn: {y.grad_fn}")
print(f"z.requires_grad: {z.requires_grad}, grad_fn: {z.grad_fn}")
print(f"z == y (值相同): {z.item() == y.item()}, 但 z 不参与梯度计算")

# no_grad: 整个块内不追踪梯度
print(f"\ntorch.no_grad():")
x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    y = x ** 2
    print(f"  no_grad 内: y.requires_grad = {y.requires_grad}")

y_outside = x ** 2
print(f"  no_grad 外: y.requires_grad = {y_outside.requires_grad}")

print("\n💡 推理时用 torch.no_grad() 节省内存 (不保存中间值)")
print("💡 detach() 常用于冻结部分网络或 target network (如 DQN)")

In [ ]:
# 8.3 Register Hook (查看/修改中间梯度)
print("\n" + "=" * 60)
print("Register Hook")
print("=" * 60)

# Tensor hook: 在反向传播时获取中间 tensor 的梯度
grad_dict = {}

def save_grad(name):
    def hook(grad):
        grad_dict[name] = grad.clone()
    return hook

x = torch.tensor(2.0, requires_grad=True)
y = x ** 3       # 中间变量
z = y ** 2       # 最终变量

# y 不是叶子节点，默认不保存梯度
y.register_hook(save_grad('y'))  # 注册 hook 保存 y 的梯度

z.backward()
print(f"  z = (x³)² = x⁶")
print(f"  dz/dx = 6x⁵ = {x.grad.item():.1f}  (理论: {6 * 2**5})")
print(f"  dz/dy = 2y = 2x³ = {grad_dict['y'].item():.1f}  (理论: {2 * 2**3})")
print(f"  (验证链式法则: dz/dx = dz/dy * dy/dx = {2*2**3} * {3*2**2} = {2*2**3 * 3*2**2})")

# retain_grad: 另一种保存非叶子梯度的方式
print("\n--- retain_grad() ---")
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.retain_grad()  # 保留 y 的梯度
z = y ** 2
z.backward()
print(f"  y.grad (retain_grad): {y.grad.item():.1f}")

In [ ]:
# 8.4 自定义 Autograd Function
print("\n" + "=" * 60)
print("自定义 Autograd Function")
print("=" * 60)

class MyReLU(torch.autograd.Function):
    """
    手写 ReLU 的前向和反向
    """
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)  # 保存输入用于反向
        return input.clamp(min=0)
    
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input < 0] = 0  # ReLU 导数: x<0 时为 0
        return grad_input

# 测试
x = torch.randn(5, requires_grad=True)
y_custom = MyReLU.apply(x)
y_official = torch.relu(x.detach().requires_grad_(True))

y_custom.sum().backward()
grad_custom = x.grad.clone()

x2 = x.detach().requires_grad_(True)
torch.relu(x2).sum().backward()
grad_official = x2.grad

print(f"  输入 x:       {x.detach().numpy().round(3)}")
print(f"  自定义梯度:   {grad_custom.numpy()}")
print(f"  官方梯度:     {grad_official.numpy()}")
print(f"  一致: {torch.equal(grad_custom, grad_official)}")

# gradcheck: 官方的梯度检查工具
from torch.autograd import gradcheck
x_check = torch.randn(5, dtype=torch.float64, requires_grad=True)
result = gradcheck(MyReLU.apply, (x_check,), eps=1e-6)
print(f"  gradcheck 通过: {result}")

---
## 9. 常见陷阱与调试技巧 <a id='9'></a>

In [ ]:
print("=" * 60)
print("常见陷阱")
print("=" * 60)

# 陷阱 1: in-place 操作破坏计算图
print("\n--- 陷阱 1: in-place 操作 ---")
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
try:
    y += 1  # in-place 操作! 等价于 y.__iadd__(1)
    y.backward()
except RuntimeError as e:
    print(f"  ✗ in-place 失败: {str(e)[:80]}...")

# 正确做法
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y = y + 1  # 非 in-place
y.backward()
print(f"  ✅ 非 in-place: grad = {x.grad.item()}")

# 陷阱 2: 叶子节点不能 in-place
print("\n--- 陷阱 2: 叶子节点 in-place ---")
x = torch.tensor(2.0, requires_grad=True)
try:
    x.add_(1)  # 叶子节点 in-place
except RuntimeError as e:
    print(f"  ✗ 叶子节点 in-place: {str(e)[:80]}...")

# 正确: 用 .data 修改 (不追踪)
x = torch.tensor(2.0, requires_grad=True)
x.data.add_(1)  # 通过 .data 绕过 autograd
print(f"  ✅ x.data.add_(1): x = {x.item()}")

# 陷阱 3: 二次 backward 需要 retain_graph
print("\n--- 陷阱 3: 二次 backward ---")
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True)  # 保留图
print(f"  第1次: grad = {x.grad.item()}")
x.grad.zero_()
y.backward()  # 第二次可以用了 (图会被释放)
print(f"  第2次: grad = {x.grad.item()}")

In [ ]:
# 内存对比: no_grad 的效果
print("\n" + "=" * 60)
print("torch.no_grad() 的内存节省")
print("=" * 60)

import tracemalloc

def measure_memory(use_grad, size=1000):
    tracemalloc.start()
    
    x = torch.randn(size, size, requires_grad=use_grad)
    if use_grad:
        y = x @ x
        z = y.sum()
    else:
        with torch.no_grad():
            y = x @ x
            z = y.sum()
    
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / (1024 * 1024)  # MB

mem_with_grad = measure_memory(True)
mem_no_grad = measure_memory(False)

print(f"  requires_grad=True:  ~{mem_with_grad:.1f} MB")
print(f"  torch.no_grad():     ~{mem_no_grad:.1f} MB")
print(f"  节省: ~{(1 - mem_no_grad/mem_with_grad)*100:.0f}%")
print("\n💡 推理时始终用 model.eval() + torch.no_grad()")

---
## 10. 思考题 & 总结 <a id='10'></a>

### 今日核心知识点

1. **Storage + Stride + View**: Tensor 的内存模型；view 零拷贝；转置只改 stride
2. **contiguous**: view 需要连续内存；reshape 自动处理；不必要的 contiguous() 会浪费
3. **计算图 DAG**: 动态图 (define-by-run)；`grad_fn` 链；反向传播后图释放
4. **链式法则**: ∂L/∂W = ∂L/∂z · ∂z/∂W；Softmax+CE 的梯度简化
5. **梯度累积**: 必须 `zero_grad()`；可用于模拟大 batch
6. **detach / no_grad**: 冻结计算 / 节省推理内存
7. **Hook**: 查看中间梯度；自定义 Autograd Function

### 面试高频问题

1. **view 和 reshape 的区别？** → view 要求 contiguous，reshape 自动处理
2. **什么是 stride？转置后 stride 怎么变？** → 交换维度的 stride
3. **PyTorch 是动态图还是静态图？** → 动态图 (define-by-run)
4. **为什么训练时要 zero_grad()？** → 梯度默认累积
5. **推理时为什么用 no_grad？** → 不建图不存中间值→省内存
6. **in-place 操作对 autograd 的影响？** → 可能破坏计算图

### 明日预习: PyTorch Module + 优化器 + 数据
- `nn.Module` 的 `forward` / `hook` / `state_dict`
- Adam / AdamW 原理
- `Dataset` / `DataLoader`
- AMP 混合精度

In [ ]:
print("\n" + "=" * 60)
print("W1 Day3 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ Tensor 创建 + dtype + NumPy 互转 (共享内存)
  ✅ Storage / Stride / View 完整实验 (切片/转置的内存模型)
  ✅ contiguous 实验 + view vs reshape 区别
  ✅ 内存布局可视化
  ✅ Autograd 计算图 DAG + grad_fn 追踪
  ✅ 常见运算的解析梯度验证 (9种)
  ✅ 手写 3 层网络反向传播 (纯 NumPy，完整训练)
  ✅ PyTorch autograd 验证手写梯度 (6组参数全部一致)
  ✅ 数值梯度检查 (有限差分法)
  ✅ 进阶: 梯度累积 / detach / no_grad / hook / 自定义 Function
  ✅ 常见陷阱: in-place / 二次 backward / 内存节省
""")